# Try Apache Beam - Python

In this notebook, we set up your development environment and work through a simple example using the [DirectRunner](https://beam.apache.org/documentation/runners/direct/). You can explore other runners with the [Beam Capatibility Matrix](https://beam.apache.org/documentation/runners/capability-matrix/).

To navigate through different sections, use the table of contents. From **View**  drop-down list, select **Table of contents**.

To run a code cell, you can click the **Run cell** button at the top left of the cell, or by select it and press **`Shift+Enter`**. Try modifying a code cell and re-running it to see what happens.

To learn more about Colab, see [Welcome to Colaboratory!](https://colab.sandbox.google.com/notebooks/welcome.ipynb).

# Rhyming Pairs in Hamlet Play - Apache Beam

# Setup

First, you need to set up your environment, which includes installing `apache-beam` and downloading a text file from Cloud Storage to your local file system. We are using this file to test your pipeline.
Also Install `Jupyter` and `Pandas` library

In [ ]:
# Run and print a shell command.
def run(cmd):
  print('>> {}'.format(cmd))
  !{cmd}
  print('')

# Install apache-beam.
# run('pip install --quiet apache-beam')
# run('pip install jupyter pandas')

# Copy the input file into the local file system.
run('mkdir -p data')
run('gsutil cp gs://dataflow-samples/shakespeare/hamlet.txt data/')

>> mkdir -p data

>> gsutil cp gs://dataflow-samples/shakespeare/midsummersnightsdream.txt data/
Copying gs://dataflow-samples/shakespeare/midsummersnightsdream.txt...
/ [1 files][ 94.2 KiB/ 94.2 KiB]                                                
Operation completed over 1 objects/94.2 KiB.                                     



# Rhyming Pairs with Comments

Below is mostly the same pipeline structure as above, but instead of counting words, we extract the last word of each line and group them by their last 3 letters to find rhyming pairs.

In [15]:
import apache_beam as beam
import re

inputs_pattern = 'data/hamlet.txt'
outputs_prefix = 'outputs/part'

# Running locally in the DirectRunner.
with beam.Pipeline() as pipeline:
  rhyming_pairs = (
      # The input PCollection is an empty pipeline.
      pipeline

      # Read lines from hamlet.txt
      | 'Read lines' >> beam.io.ReadFromText(inputs_pattern)
      # Element type: str - text line

      # Extract the last word of each line
      | 'Get last word' >> beam.FlatMap(lambda line: [re.findall(r"[a-zA-Z]+", line)[-1].lower()] if re.findall(r"[a-zA-Z]+", line) else [])
      # Element type: str - last word of line

      # Keep only words 4+ letters so short words don't pollute results
      | 'Filter short words' >> beam.Filter(lambda word: len(word) >= 4)
      # Element type: str - filtered word

      # Use last 3 letters as rhyme key
      | 'Extract rhyme key' >> beam.Map(lambda word: (word[-3:], word))
      # Element type: (str, str) - key: rhyme ending, value: word

      # Group words by rhyme ending
      | 'Group by ending' >> beam.GroupByKey()
      # Element type: (str, [str]) - key: rhyme ending, value: list of words

      # Keep only groups with 2+ unique words
      | 'Filter singles' >> beam.Filter(lambda x: len(set(x[1])) >= 2)
      # Element type: (str, [str]) - rhyme groups with 2+ words

      # Format as readable output showing ending and unique rhyming words
      | 'Format results' >> beam.Map(lambda x: str((x[0], sorted(set(x[1])))))
      # Element type: str - text line

      # Write to output file
      | 'Write results' >> beam.io.WriteToText(outputs_prefix)
  )

# Display results
run('cat {}-00000-of-*'.format(outputs_prefix))

>> cat outputs/part-00000-of-*
('ius', ['claudius', 'cornelius', 'polonius'])
('ing', ['agreeing', 'asking', 'breeding', 'coming', 'contriving', 'crawling', 'dallying', 'drooping', 'fighting', 'finding', 'following', 'hearing', 'howling', 'king', 'lasting', 'living', 'making', 'marching', 'meeting', 'morning', 'nothing', 'packing', 'playing', 'poisoning', 'praying', 'quarrelling', 'reading', 'ring', 'seeing', 'seeming', 'showing', 'sipping', 'smiling', 'spring', 'stirring', 'suiting', 'thing', 'warning', 'wedding', 'whipping', 'wing', 'writing'])
('and', ['command', 'england', 'hand', 'land', 'poland', 'stand', 'thousand', 'voltimand'])
('ers', ['brokers', 'brothers', 'carters', 'courtiers', 'diggers', 'enters', 'fingers', 'flowers', 'hangers', 'letters', 'makers', 'matters', 'messengers', 'numbers', 'officers', 'others', 'players', 'prayers', 'recorders', 'showers', 'slaughters', 'soldiers', 'stealers', 'tenders', 'vouchers', 'whispers'])
('men', ['amen', 'countrymen', 'gentlemen'])
(